# Extended tests — scale, real forecasting corpora, and a dissociation

Companion to `marv_titans_temporal_colab.ipynb`. That notebook runs the *controlled*
sweep (marginals matched, only predictability moves). This one runs the three tests that
extend what the controlled result is allowed to claim:

| Test | Question it answers | Why it matters |
|---|---|---|
| **A · Memory scale** | does the behaviour survive past 256 units? | direct answer to "this is toy scale" |
| **B · Real TSFM corpora** | does it hold on the data foundation models pretrain on? | external validity |
| **C · Protein dissociation** | is it *local predictability* or *structure exists*? | discriminates two mechanisms |

**Read the caveats.** Test A is a clean single-variable sweep. Test B corpora are
marginal-matched to each other via quantile binning, so they are comparable. Test C is
**not** marginal-matched (amino acids are categorical, like text bytes) -- it is a
dissociation probe, not a controlled point.

## Setup

In [ ]:
!pip install -q titans-pytorch datasets
!git clone -q https://github.com/thebnbrkr/marv-titan.git /content/marv-titan
import sys; sys.path.insert(0, '/content/marv-titan/experiments')

import os, json, time, urllib.request
import numpy as np, torch
import matplotlib.pyplot as plt
import pandas as pd

# --- the SAME code as the baseline runs, imported from the repo ---
from titans_real_text import (
    build_model, sample_batch, train, _cos,
    consecutive_write_alignment, DIM_HEAD, HIDDEN,
)
from titans_pytorch import MemoryAsContextTransformer, MemoryMLP   # only for build_model_scaled

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
N_BINS   = 256
STEPS, SEQ_LEN, BATCH, LR = 2000, 256, 8, 2e-4
SEEDS = [0, 1, 2]
PASSAGE_LEN = 1024
print("device:", DEVICE, "| repo memory:", DIM_HEAD, "->", HIDDEN, "->", DIM_HEAD)

In [ ]:
# --- helpers that are NOT in the repo (binning + information measures) ----------
def quantize(series, n_bins=N_BINS, split=0.9):
    s = np.asarray(series, dtype=np.float64); s = s[np.isfinite(s)]
    k = int(len(s) * split)
    edges = np.quantile(s[:k], np.linspace(0, 1, n_bins + 1)[1:-1])
    return (torch.from_numpy(np.digitize(s[:k], edges)).long(),
            torch.from_numpy(np.digitize(s[k:], edges)).long())

def marginal_entropy_nats(tok, n_bins=N_BINS):
    c = np.bincount(tok.numpy(), minlength=n_bins).astype(float); p = c / c.sum(); p = p[p > 0]
    return float(-(p * np.log(p)).sum())

def lag1_mi(tok, coarse=16):
    """Lag-1 mutual information (nats): how much the previous symbol tells you about
    the next. Coarsens by the ACTUAL alphabet size -- a fixed /256 collapses a
    20-symbol protein stream onto 2 bins and reports ~0 regardless of the truth.
    Small alphabets are histogrammed directly, where counts are already dense.
    Bias floor is ~(bins^2)/(2N) nats; treat values under 0.001 as zero."""
    x = tok.numpy().astype(int)
    n_sym = int(x.max()) + 1
    bins = n_sym if n_sym <= 2 * coarse else coarse
    if bins < n_sym:
        x = (x * bins) // n_sym
    J = np.histogram2d(x[:-1], x[1:], bins=[bins, bins])[0]; J = J / J.sum()
    px, py = J.sum(1, keepdims=True), J.sum(0, keepdims=True); m = J > 0
    return float((J[m] * np.log(J[m] / (px @ py)[m])).sum())


# --- the ONE deviation from the repo: a width-parameterised build_model ---------
# The repo's build_model hardcodes expansion_factor=4.0 (64 -> 256 -> 64). The scale
# sweep needs to vary that one number. Every other argument is copied verbatim; the
# next cell asserts that build_model_scaled(256) matches the repo's build_model()
# parameter-for-parameter, so the 256-unit rung is a true baseline.
def build_model_scaled(mem_hidden=HIDDEN, neural_memory_segment_len: int = 8):
    return MemoryAsContextTransformer(
        num_tokens=256, dim=DIM_HEAD, depth=4, segment_len=32,
        neural_memory_segment_len=neural_memory_segment_len,
        num_persist_mem_tokens=4, num_longterm_mem_tokens=4,
        neural_memory_layers=(2,), dim_head=32, heads=2,
        neural_memory_model=MemoryMLP(DIM_HEAD, depth=2,
                                      expansion_factor=mem_hidden / DIM_HEAD),
        neural_memory_kwargs=dict(dim_head=DIM_HEAD, heads=1), use_flex_attn=False)


@torch.no_grad()
def val_loss(model, va, n=20):
    model.eval()
    return float(np.mean([model(sample_batch(va, SEQ_LEN, BATCH).to(DEVICE),
                                return_loss=True).item() for _ in range(n)]))

@torch.no_grad()
def measure_memory(model, passage):
    """Same computation as the repo's diff_memory_on_passage, returning instead of
    printing; write-alignment comes from the repo's own function."""
    model.eval()
    _, cache = model(passage.unsqueeze(0).to(DEVICE), return_cache=True)
    state = cache[2][0]
    U0 = state.updates["model.weights.0"].detach()[0].cpu().numpy()
    g_in, g_out = U0[1], U0[-1]
    gate_cos = _cos(g_in, g_out, axis=0)
    nr = (np.linalg.norm(g_out, axis=0) + 1e-9) / (np.linalg.norm(g_in, axis=0) + 1e-9)
    moved = gate_cos < 0.99
    consec = consecutive_write_alignment(model, passage, DEVICE)
    return {"norm_ratio": float(nr[moved].mean()) if moved.any() else float("nan"),
            "units_moved": int(moved.sum()), "n_units": int(len(gate_cos)),
            "write_align": float(consec.mean())}

In [ ]:
# Assert the width-parameterised builder reproduces the repo's model at 256 units.
_a, _b = build_model(), build_model_scaled(HIDDEN)
_sa = {k: tuple(v.shape) for k, v in _a.state_dict().items()}
_sb = {k: tuple(v.shape) for k, v in _b.state_dict().items()}
assert _sa == _sb, [k for k in set(_sa) | set(_sb) if _sa.get(k) != _sb.get(k)]
print(f"build_model_scaled({HIDDEN}) matches the repo's build_model(): "
      f"{len(_sa)} tensors, identical shapes")
del _a, _b

# Confirm the width knob actually moves the memory weights (no hardcoded key -- look
# for tensors whose shape carries the requested hidden width).
_prev = None
for _h in (256, 512, 1024):
    _m = build_model_scaled(_h)
    _hits = {k: tuple(v.shape) for k, v in _m.state_dict().items()
             if v.ndim >= 2 and _h in tuple(v.shape)}
    _n_params = sum(v.numel() for v in _m.state_dict().values())
    _example = next(iter(_hits.items()), ("<none found>", ()))
    print(f"  mem_hidden {_h:>5} -> {len(_hits):>2} tensors carry dim {_h}"
          f" | e.g. {_example[0]} {_example[1]} | total params {_n_params:,}")
    assert _hits, f"no tensor has a {_h} dimension -- expansion_factor did not take effect"
    if _prev is not None:
        assert _n_params > _prev, "widening the memory did not increase parameter count"
    _prev = _n_params
    del _m
print("width knob verified: memory grows with mem_hidden")

## Test A · Does it survive past 256 units?

The single most common objection to this work is "256 hidden units is a toy." This is the
cheapest possible answer: widen **only** the memory MLP (64 -> N -> 64) and change nothing
else. Same data, same depth, same everything. One variable.

If `norm_ratio` behaves the same at 1024 units as at 256, the toy-scale objection loses
most of its force. If it *changes*, that is itself worth reporting -- it would mean the
retention regime is capacity-dependent.

In [ ]:
def ar1(n, phi, seed=0):
    rng = np.random.default_rng(seed); e = rng.standard_normal(n); x = np.zeros(n)
    for t in range(1, n): x[t] = phi * x[t - 1] + e[t]
    return x

SCALE_SIZES = [256, 512, 1024]
SCALE_CORPORA = {"ar_phi0.00": 0.0, "ar_phi0.90": 0.9}

scale_rows = []
for name, phi in SCALE_CORPORA.items():
    tr, va = quantize(ar1(400_000, phi))
    for h in SCALE_SIZES:
        for seed in SEEDS:
            torch.manual_seed(seed); np.random.seed(seed)
            print(f"\n=== {name} | {h} units | seed {seed} ===")
            t0 = time.time()
            m = build_model_scaled(mem_hidden=h).to(DEVICE)
            train(m, tr, va, STEPS, SEQ_LEN, BATCH, LR, DEVICE)
            row = {"corpus": name, "mem_units": h, "seed": seed,
                   "val_loss": val_loss(m, va), "minutes": (time.time() - t0) / 60}
            row.update(measure_memory(m, va[:PASSAGE_LEN]))
            print(f"  norm_ratio {row['norm_ratio']:.2f} | moved {row['units_moved']}/{row['n_units']}"
                  f" | val {row['val_loss']:.3f} | {row['minutes']:.1f} min")
            scale_rows.append(row)
            del m; torch.cuda.empty_cache()
            json.dump(scale_rows, open("scale_results.json", "w"), indent=2)

pd.DataFrame(scale_rows).groupby(["corpus", "mem_units"])[["norm_ratio", "val_loss"]].mean().round(3)

## Test B · Real corpora that temporal foundation models pretrain on

`autogluon/chronos_datasets` (67 configs) and `Salesforce/lotsa_data` (174 configs) are the
pretraining corpora behind Chronos and Moirai. Streaming avoids downloading the full
archives. Each series is quantised with its **own** edges, then concatenated -- so every
corpus here lands at the same marginal entropy and is comparable to the AR sweep.

In [ ]:
from datasets import load_dataset

def _numeric_seq_column(row):
    """Find the column holding the actual series -- schemas differ across configs."""
    best, best_len = None, 0
    for k, v in row.items():
        if isinstance(v, (list, np.ndarray)) and len(v) > best_len:
            try:
                float(v[0]); best, best_len = k, len(v)
            except (TypeError, ValueError, IndexError):
                pass
    return best

def load_hf_series(repo, config, max_series=10_000, max_points=400_000):
    """Stream a HF time-series dataset -> (train_tokens, val_tokens), quantised
    per-series so no single series' scale dominates the binning."""
    ds = load_dataset(repo, config, split="train", streaming=True)
    tr_parts, va_parts, total = [], [], 0
    col = None
    for i, row in enumerate(ds):
        if col is None:
            col = _numeric_seq_column(row)
            if col is None: raise ValueError(f"no numeric sequence column in {repo}/{config}")
            print(f"  using column '{col}'")
        s = np.asarray(row[col], dtype=np.float64)
        s = s[np.isfinite(s)]
        if len(s) < 200: continue
        t, v = quantize(s)
        tr_parts.append(t); va_parts.append(v); total += len(s)
        if total >= max_points or len(tr_parts) >= max_series: break
    if not tr_parts: raise ValueError("no usable series found")
    print(f"  {len(tr_parts)} series, {total:,} points")
    return torch.cat(tr_parts), torch.cat(va_parts)


HF_CORPORA = {
    "chronos/m4_hourly":       ("autogluon/chronos_datasets", "m4_hourly"),
    "chronos/electricity_15min":("autogluon/chronos_datasets", "electricity_15min"),
    "chronos/exchange_rate":   ("autogluon/chronos_datasets", "exchange_rate"),
    "lotsa/PEMS08":            ("Salesforce/lotsa_data", "PEMS08"),
}

hf_rows = []
for name, (repo, cfg) in HF_CORPORA.items():
    for seed in SEEDS:
        print(f"\n=== {name} | seed {seed} ===")
        try:
            tr, va = load_hf_series(repo, cfg)
        except Exception as e:
            print(f"  SKIPPED ({type(e).__name__}: {e})"); continue
        torch.manual_seed(seed); np.random.seed(seed)
        t0 = time.time()
        m = build_model().to(DEVICE)
        train(m, tr, va, STEPS, SEQ_LEN, BATCH, LR, DEVICE)
        row = {"corpus": name, "seed": seed, "val_loss": val_loss(m, va),
               "marg_entropy": marginal_entropy_nats(tr), "lag1_mi": lag1_mi(tr),
               "minutes": (time.time() - t0) / 60}
        row.update(measure_memory(m, va[:PASSAGE_LEN]))
        print(f"  norm_ratio {row['norm_ratio']:.2f} | MI {row['lag1_mi']:.3f}"
              f" | entropy {row['marg_entropy']:.3f} | val {row['val_loss']:.3f}")
        hf_rows.append(row); del m; torch.cuda.empty_cache()
        json.dump(hf_rows, open("hf_results.json", "w"), indent=2)

pd.DataFrame(hf_rows).round(3) if hf_rows else print("no HF corpora completed")

## Test C · The dissociation — protein sequences

Every corpus so far tangles two things together: **local predictability** and **the
existence of learnable structure**. Text has both. Noise has neither. Protein has the
second without much of the first -- the next amino acid is famously hard to predict from
the previous one, yet real long-range structure (motifs, domains) exists.

So protein splits the hypothesis:

- memory **forgets** on protein -> retention tracks *local* predictability (the `lag1_mi` axis)
- memory **accumulates** on protein -> retention tracks *structure being present*, and local
  predictability was only a proxy

**Caveat, same as enwik8:** amino acids are categorical, so no quantile binning and the
marginal is *not* matched (~log 20 = 3.0 nats, not 5.545). This is a dissociation probe,
not a point in the controlled sweep. Report it as such.

In [ ]:
AA = "ACDEFGHIKLMNPQRSTVWY"   # 20 standard amino acids
AA_MAP = {c: i for i, c in enumerate(AA)}

def load_protein(max_residues=400_000):
    ds = load_dataset("agemagician/uniref50", split="train", streaming=True)
    buf, col = [], None
    for row in ds:
        if col is None:
            col = next((k for k, v in row.items()
                        if isinstance(v, str) and len(v) > 30
                        and sum(c in AA_MAP for c in v[:50]) > 40), None)
            if col is None: raise ValueError(f"no sequence column; keys={list(row.keys())}")
            print(f"  using column '{col}'")
        buf.extend(AA_MAP[c] for c in row[col] if c in AA_MAP)
        if len(buf) >= max_residues: break
    x = np.array(buf[:max_residues], dtype=np.int64)
    k = int(len(x) * 0.9)
    print(f"  {len(x):,} residues")
    return torch.from_numpy(x[:k]), torch.from_numpy(x[k:])

prot_rows = []
for seed in SEEDS:
    print(f"\n=== protein (uniref50) | seed {seed} ===")
    try:
        tr, va = load_protein()
    except Exception as e:
        print(f"  SKIPPED ({type(e).__name__}: {e})"); break
    torch.manual_seed(seed); np.random.seed(seed)
    m = build_model().to(DEVICE)
    train(m, tr, va, STEPS, SEQ_LEN, BATCH, LR, DEVICE)
    row = {"corpus": "protein", "seed": seed, "val_loss": val_loss(m, va),
           "marg_entropy": marginal_entropy_nats(tr), "lag1_mi": lag1_mi(tr)}
    row.update(measure_memory(m, va[:PASSAGE_LEN]))
    print(f"  norm_ratio {row['norm_ratio']:.2f} | MI {row['lag1_mi']:.4f}"
          f" | entropy {row['marg_entropy']:.3f} (log 20 = {np.log(20):.3f})")
    prot_rows.append(row); del m; torch.cuda.empty_cache()
    json.dump(prot_rows, open("protein_results.json", "w"), indent=2)

pd.DataFrame(prot_rows).round(4) if prot_rows else print("protein test did not run")

## Reading the extended results

**Test A (scale).** Plot `norm_ratio` against `mem_units` for each corpus. Flat = the
finding is not a 256-unit artifact, and you can say so explicitly in limitations. Sloped =
capacity matters, which is a finding in its own right and needs saying.

**Test B (real corpora).** These are marginal-matched to the AR sweep, so they belong on the
same axis. If the AR trend predicts where they land, that is strong external validity: a
relationship found on synthetic data holding on the corpora real forecasting models are
trained on.

**Test C (protein).** The discriminating one. Compare its `norm_ratio` against `ar_phi0.00`
(no structure) and text (structure + high local predictability). Which one it resembles
tells you which mechanism is doing the work -- and either answer is publishable.

**What none of this licenses.** Even with all three, the claim stays bounded to: one
architecture, one unofficial implementation, small memories, short training. Scale and
corpus breadth widen the evidence; they do not turn it into a statement about production
systems. Keep the claim one notch weaker than the evidence feels.